## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, concat, upper, trim, lower, regexp_replace, substring, when

## Reading from bronze layer 

In [0]:
df = spark.table("olist.bronze.products")
df.display()

## Overview about products table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()



## Transformations

### 1. TRIM whitespace from all string columns

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))


### 2. Normalize improperly represented nulls in string columns 

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-", " "]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

###  3. Normalize zeros in numeric columns where 0 is not a valid value

In [0]:
numeric_cols_no_zero = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
for c in numeric_cols_no_zero:
    df = df.withColumn(
        c,
        when(col(c) == 0, None).otherwise(col(c))
    )

### 4. Standardize product_category_name - replace underscores with spaces & lowercase

In [0]:
df = df.withColumn(
    "product_category_name",
    lower(regexp_replace(col("product_category_name"), "_", " "))
)

### 5. Fill nulls in product_category_name column

In [0]:
df = df.fillna({"product_category_name": "unknown"})

### 6. Rename misspelled columns

In [0]:
df = df.withColumnRenamed("product_name_lenght", "product_name_length") \
       .withColumnRenamed("product_description_lenght", "product_description_length")

### 7. Handle nulls - filter out rows missing critical key

In [0]:
df = df.filter(col("product_id").isNotNull())

### 8. Remove duplicates on product_id

In [0]:
df = df.dropDuplicates(["product_id"])

## Quality Checks

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Unique products: {df.select('product_id').distinct().count()}")
print(f"Null product_category_name: {df.filter(col('product_category_name').isNull()).count()}")
print(f"Null product_weight_g: {df.filter(col('product_weight_g').isNull()).count()}")
print(f"Null product_length_cm: {df.filter(col('product_length_cm').isNull()).count()}")
print(f"Null product_height_cm: {df.filter(col('product_height_cm').isNull()).count()}")
print(f"Null product_width_cm: {df.filter(col('product_width_cm').isNull()).count()}")
df.display()

## Write to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.products")